# [Encuesta de satisfacción en Google Form](https://forms.gle/YUc83ygwGoEcHgM2A) 

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marco-canas/intro_ml_dl/blob/main/2_planificacion/3_dl/geron/10_chapter/4_pagina_485_regression_with_mlp/pag_485_regression_with_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marco-canas/intro_ml_dl/blob/main/2_planificacion/3_dl/geron/10_chapter/4_pagina_485_regression_with_mlp/pag_485_regression_with_mlp.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

### [Video de apoyo a la lectura interactiva y experimental de este cuaderno]()

### [Vínculo al programa del curso Deep Learning par Administración de empresas: ]()



Un reconocimiento a mis estudiantes que han construido conmigo este saber pedagógico:

<img src = 'https://github.com/marco-canas/intro_ml_dl/blob/main/5_images/taller_estructuras_datos_python_2025-09-04%20a%20las%2020.38.jpg?raw=true'> 



In [1]:
import pandas as pd
import numpy as np

# Cargar la lista de estudiantes desde el archivo CSV
path = 'C:/Users/marco/Documentos/docencia/groups_list/g_lideres.xlsx'
df = pd.read_excel(path)

df.head(3)

,Nombre,Programa
0,marco julio cañas campillo,licenciatura en matemáticas



## Resumen explicado en español del fragmento (págs. 485–487)



**MLPs para regresión**



* Una red neuronal de tipo *perceptrón multicapa* (MLP) puede utilizarse no solo para clasificación, sino también para **regresión**.
* Si queremos predecir un solo valor (ejemplo: el precio de una casa), basta con **una neurona de salida**.
* Para regresión multivariada (varios valores a la vez), se usa **una neurona por dimensión de salida**. Ejemplo:

  * Centro de un objeto en 2D → 2 neuronas (coordenadas).
  * Caja delimitadora (ancho y alto) → 2 neuronas adicionales.
  * Total: 4 neuronas de salida.



**Ejemplo en Scikit-Learn**

* Scikit-Learn tiene la clase `MLPRegressor`.
* Ejemplo con el dataset de viviendas de California (`fetch_california_housing`).
* Se construye un MLP con **3 capas ocultas de 50 neuronas cada una**.
* Se usa un *pipeline* con `StandardScaler` para escalar los datos (muy importante para que el descenso de gradiente converja).
* Entrenamiento con función de activación ReLU en las capas ocultas, y optimizador **Adam** para minimizar el error cuadrático medio (MSE).
* Resultado: un RMSE de validación ≈ **0.505**, comparable al obtenido con un *random forest*.



**Funciones de activación en la salida**

* Por defecto, la capa de salida no tiene función de activación (puede devolver cualquier valor).
* Si se requieren solo valores positivos → usar ReLU o *softplus*.
* Si se necesitan salidas en un rango acotado → usar sigmoide (0–1) o tangente hiperbólica (–1 a 1).
* Limitación: `MLPRegressor` de Scikit-Learn no soporta funciones de activación en la salida.



**Sobre la función de pérdida**

* `MLPRegressor` solo soporta **MSE**.
* Pero en algunos casos puede ser más útil:

  * **MAE (error absoluto medio)**: más robusto ante *outliers*.
  * **Huber loss**: combina MSE y MAE; es cuadrática para errores pequeños (precisión) y lineal para errores grandes (robustez).



**Arquitectura típica de un MLP para regresión (Tabla 10-1)**

* Capas ocultas: 1 a 5.
* Neuronas por capa: entre 10 y 100.
* Neuronas de salida: una por cada variable a predecir.
* Activación oculta: ReLU.
* Activación de salida: ninguna, o ReLU/softplus (si salidas positivas), sigmoide/tanh (si salidas acotadas).
* Función de pérdida: MSE, o Huber si hay *outliers*.



# Guía práctica: MLPs para regresión en Scikit-Learn


In [2]:

# ================================================================
#   PERCEPTRÓN MULTICAPA (MLP) PARA REGRESIÓN
#   Ejemplo con el dataset de viviendas de California
#   Autor: Marco Julio (adaptado de Hands-On Machine Learning)
# ================================================================

# 1. Importar librerías necesarias
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 2. Cargar el dataset de California
housing = fetch_california_housing()
X, y = housing.data, housing.target

print("Características del dataset:")
print(housing.feature_names)
print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# 3. Dividir en entrenamiento, validación y prueba
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

print("Tamaño del conjunto de entrenamiento:", X_train.shape[0])
print("Tamaño del conjunto de validación:", X_valid.shape[0])
print("Tamaño del conjunto de prueba:", X_test.shape[0])

# 4. Crear el modelo MLP con tres capas ocultas de 50 neuronas
mlp_reg = MLPRegressor(
    hidden_layer_sizes=[50, 50, 50],  # 3 capas ocultas con 50 neuronas cada una
    activation="relu",                # función de activación ReLU
    solver="adam",                    # optimizador Adam
    alpha=0.0001,                     # regularización L2 (para evitar sobreajuste)
    max_iter=500,                     # número máximo de iteraciones
    random_state=42
)

# 5. Pipeline: escalado de variables + red neuronal
pipeline = make_pipeline(StandardScaler(), mlp_reg)

# 6. Entrenar el modelo
pipeline.fit(X_train, y_train)

# 7. Evaluación en el conjunto de validación
y_pred = pipeline.predict(X_valid)
rmse = mean_squared_error(y_valid, y_pred, squared=False)

print(f"Error cuadrático medio de validación (RMSE): {rmse:.3f}")

# 8. Evaluación final en el conjunto de prueba
y_pred_test = pipeline.predict(X_test)
rmse_test = mean_squared_error(y_test, y_pred_test, squared=False)

print(f"Error cuadrático medio en el conjunto de prueba (RMSE): {rmse_test:.3f}")



Características del dataset:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dimensiones de X: (20640, 8)
Dimensiones de y: (20640,)
Tamaño del conjunto de entrenamiento: 13209
Tamaño del conjunto de validación: 3303
Tamaño del conjunto de prueba: 4128


TypeError: got an unexpected keyword argument 'squared'


---

## ✨ Explicación paso a paso

1. **Dataset**:

   * Usamos `fetch_california_housing`, que contiene precios medios de casas en distritos de California con variables como: número de habitaciones, población, etc.
   * Es limpio: solo variables numéricas y sin valores perdidos.

2. **Preprocesamiento**:

   * Escalamos las variables con `StandardScaler`.
   * Esto es crucial porque el descenso de gradiente converge mal si las variables están en escalas muy diferentes.

3. **Modelo MLP**:

   * Tres capas ocultas con 50 neuronas cada una.
   * Función de activación: **ReLU**.
   * Optimizador: **Adam**.
   * Regularización ℓ₂ (`alpha`) para controlar sobreajuste.

4. **Entrenamiento y validación**:

   * El RMSE en validación es cercano a **0.5** (en miles de dólares, ya que el objetivo está en esa escala).
   * Comparado con un bosque aleatorio (*random forest*), es competitivo.

5. **Interpretación**:

   * El MLP aprende relaciones no lineales entre las variables y el precio.
   * Ajustando hiperparámetros (número de capas, neuronas, `alpha`, `learning_rate_init`) se puede mejorar el rendimiento.

---

👉 Si quieres, te preparo una **versión interactiva con `ipywidgets`** para que los estudiantes puedan experimentar cambiando el número de capas, neuronas y el `alpha` y ver cómo cambia el RMSE.

¿Quieres que te arme esa versión interactiva también?


¡Excelente! 🚀
Aquí tienes la **versión interactiva con `ipywidgets`** para tu cuaderno Jupyter. Tus estudiantes podrán cambiar hiperparámetros (capas ocultas, neuronas, regularización) y ver cómo cambia el rendimiento del modelo MLP en el dataset de California.

---



## 📓 MLP para regresión con controles interactivos

```python
# ================================================================
#   MLP para regresión con controles interactivos
#   Usando ipywidgets para explorar hiperparámetros
# ================================================================

import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from ipywidgets import interact

# 1. Cargar dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target

# 2. Dividir en entrenamiento, validación y prueba
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

# 3. Función para entrenar y evaluar el modelo
def entrenar_mlp(n_capas=2, n_neuronas=50, alpha=0.0001, max_iter=300):
    # Definir estructura de la red
    hidden_layers = tuple([n_neuronas] * n_capas)
    
    mlp_reg = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        alpha=alpha,
        max_iter=max_iter,
        random_state=42
    )
    
    pipeline = make_pipeline(StandardScaler(), mlp_reg)
    pipeline.fit(X_train, y_train)
    
    # Evaluación
    y_pred = pipeline.predict(X_valid)
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    
    y_pred_test = pipeline.predict(X_test)
    rmse_test = mean_squared_error(y_test, y_pred_test, squared=False)
    
    # Mostrar resultados
    print(f"Arquitectura oculta: {hidden_layers}")
    print(f"Regularización (alpha): {alpha}")
    print(f"Iteraciones: {max_iter}")
    print(f"RMSE validación: {rmse:.3f}")
    print(f"RMSE prueba: {rmse_test:.3f}")
    
    # Comparar predicción vs valores reales
    plt.figure(figsize=(6, 6))
    plt.scatter(y_valid, y_pred, alpha=0.3)
    plt.plot([y_valid.min(), y_valid.max()],
             [y_valid.min(), y_valid.max()],
             "r--", lw=2)
    plt.xlabel("Valores reales")
    plt.ylabel("Predicciones")
    plt.title("Predicciones vs Reales (Validación)")
    plt.show()

# 4. Crear controles interactivos
interact(
    entrenar_mlp,
    n_capas=widgets.IntSlider(min=1, max=5, step=1, value=3, description="Capas ocultas"),
    n_neuronas=widgets.IntSlider(min=10, max=200, step=10, value=50, description="Neuronas"),
    alpha=widgets.FloatLogSlider(base=10, min=-5, max=0, step=0.1, value=0.0001, description="Alpha"),
    max_iter=widgets.IntSlider(min=100, max=1000, step=100, value=300, description="Iteraciones")
)
```

---

## ✨ Cómo usarlo

1. Corre la celda en Jupyter Notebook.
2. Ajusta los **deslizadores**:

   * Número de capas ocultas (1–5).
   * Número de neuronas por capa (10–200).
   * Valor de regularización `alpha` (10⁻⁵ a 1).
   * Iteraciones de entrenamiento.
3. Observa en consola los valores de **RMSE** en validación y prueba.
4. Revisa la **gráfica** que compara los valores reales vs. las predicciones.

---

¿Quieres que le agregue también una **sección de experimento guiado** (con preguntas para tus estudiantes, como “¿qué pasa si aumentamos capas?” o “¿qué ocurre al aumentar `alpha`?”) para integrarlo como actividad didáctica en clase?


¡Perfecto, Marco! 🙌
Aquí tienes la **versión con experimento guiado** para tu cuaderno Jupyter. La idea es que los estudiantes no solo jueguen con los parámetros, sino que también respondan preguntas reflexivas que los lleven a entender **qué ocurre con el MLP cuando se modifican los hiperparámetros**.

---

## 📓 MLP para regresión con experimento guiado

```python
# ================================================================
#   MLP para regresión con controles interactivos + experimento guiado
#   Curso: Análisis de Datos / Machine Learning
#   Actividad práctica para estudiantes
# ================================================================

import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from ipywidgets import interact

# 1. Cargar dataset
housing = fetch_california_housing()
X, y = housing.data, housing.target

# 2. Dividir en entrenamiento, validación y prueba
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

# 3. Función para entrenar y evaluar el modelo
def entrenar_mlp(n_capas=2, n_neuronas=50, alpha=0.0001, max_iter=300):
    hidden_layers = tuple([n_neuronas] * n_capas)
    
    mlp_reg = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        alpha=alpha,
        max_iter=max_iter,
        random_state=42
    )
    
    pipeline = make_pipeline(StandardScaler(), mlp_reg)
    pipeline.fit(X_train, y_train)
    
    # Evaluación
    y_pred = pipeline.predict(X_valid)
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    
    y_pred_test = pipeline.predict(X_test)
    rmse_test = mean_squared_error(y_test, y_pred_test, squared=False)
    
    # Resultados en consola
    print(f"Arquitectura oculta: {hidden_layers}")
    print(f"Regularización (alpha): {alpha}")
    print(f"Iteraciones: {max_iter}")
    print(f"RMSE validación: {rmse:.3f}")
    print(f"RMSE prueba: {rmse_test:.3f}")
    
    # Gráfica
    plt.figure(figsize=(6, 6))
    plt.scatter(y_valid, y_pred, alpha=0.3)
    plt.plot([y_valid.min(), y_valid.max()],
             [y_valid.min(), y_valid.max()],
             "r--", lw=2)
    plt.xlabel("Valores reales")
    plt.ylabel("Predicciones")
    plt.title("Predicciones vs Reales (Validación)")
    plt.show()

# 4. Controles interactivos
interact(
    entrenar_mlp,
    n_capas=widgets.IntSlider(min=1, max=5, step=1, value=3, description="Capas ocultas"),
    n_neuronas=widgets.IntSlider(min=10, max=200, step=10, value=50, description="Neuronas"),
    alpha=widgets.FloatLogSlider(base=10, min=-5, max=0, step=0.1, value=0.0001, description="Alpha"),
    max_iter=widgets.IntSlider(min=100, max=1000, step=100, value=300, description="Iteraciones")
)
```

---

## 🧪 Experimento guiado para los estudiantes

1. **Explora la profundidad del modelo**

   * Fija `alpha = 0.0001` y `neuronas = 50`.
   * Varía el número de **capas ocultas** de 1 a 5.
   * Pregunta:

     * ¿Qué ocurre con el **RMSE de validación** al aumentar las capas?
     * ¿El modelo mejora siempre o empieza a sobreajustar?

2. **Explora la capacidad de cada capa**

   * Fija `capas = 2`.
   * Varía el número de **neuronas** (de 10 a 200).
   * Pregunta:

     * ¿Qué efecto tiene en el **error de validación**?
     * ¿Qué pasa con el tiempo de entrenamiento?

3. **Explora la regularización**

   * Fija `capas = 3`, `neuronas = 100`.
   * Varía **alpha** de `1e-5` a `1`.
   * Pregunta:

     * ¿Qué pasa con el RMSE cuando `alpha` es muy bajo?
     * ¿Y cuando `alpha` es muy alto?
     * ¿Qué interpretación le darías en términos de **sobreajuste vs subajuste**?

4. **Explora el número de iteraciones**

   * Fija `capas = 3`, `neuronas = 50`, `alpha = 0.001`.
   * Varía **max\_iter** de 100 a 1000.
   * Pregunta:

     * ¿Qué pasa si el modelo no converge en pocas iteraciones?
     * ¿Siempre mejora al aumentar el número de iteraciones?

---

✅ Con esto tienes un **laboratorio interactivo de aprendizaje**.
Los estudiantes pueden experimentar, observar métricas y responder las preguntas como parte de una actividad evaluativa.

---

¿Quieres que también te prepare una **rúbrica de evaluación** para calificar esta actividad en tu curso (por ejemplo, puntaje por interpretación, experimentación y redacción de conclusiones)?


La **lectura interactiva y experimental** de los cuadernos Jupyter diseñados para el curso de **Fundamentos de Lógica** implica un enfoque dinámico y práctico para el aprendizaje, donde los estudiantes no solo consumen información teórica, sino que también interactúan con el contenido, modifican ejemplos, ejecutan código y experimentan con los conceptos lógicos en un entorno computacional. A continuación, se detallan las características clave de este enfoque:

---



### **1. Lectura Interactiva**  
- **Manipulación directa del contenido**: Los estudiantes pueden ejecutar celdas de código, modificar fórmulas lógicas o ejemplos, y observar cómo cambian los resultados en tiempo real.  
- **Visualización interactiva**: Uso de gráficos, diagramas (como árboles semánticos o tablas de verdad) o herramientas que respondan a entradas del usuario para ilustrar conceptos como validez, consistencia o inferencia.  
- **Retroalimentación inmediata**: Los cuadernos pueden incluir ejercicios con autoevaluación (ejecutando código que verifica soluciones) o explicaciones emergentes al resolver problemas.  

---



### **2. Lectura Experimental**  
- **Aprendizaje basado en prueba y error**: Los estudiantes pueden "jugar" con estructuras lógicas (por ejemplo, modificar conectores en una fórmula proposicional y ver cómo afecta su tabla de verdad).  
- **Simulación de escenarios**: Por ejemplo, modelar argumentos en lógica de primer orden y evaluar su corrección mediante ejecución de código (usando librerías como `sympy` o herramientas ad-hoc).  
- **Exploración guiada y abierta**: Se incluyen secciones con consignas del tipo *"¿Qué pasa si cambias este axioma?"* o *"Intenta construir un contraejemplo"* para fomentar la curiosidad.  

---



### **3. Componentes clave de los cuadernos**  
- **Fragmentos de código ejecutable**: Para evaluar expresiones lógicas, automatizar pruebas o implementar algoritmos (ej: verificación de tautologías).  
- **Celdas con texto teórico y preguntas reflexivas**: Integradas con ejemplos prácticos que requieren intervención activa (ej: *"Define aquí tu propia fórmula y comprueba si es satisfacible"*).  
- **Enlaces a recursos externos**: Como demostradores en línea o lecturas complementarias para profundizar.  

---



### **4. Beneficios pedagógicos**  
- **Enganche activo**: Combina teoría y práctica sin salir del entorno digital.  
- **Personalización**: Los estudiantes pueden ajustar el ritmo y profundidad de su aprendizaje.  
- **Preparación para aplicaciones reales**: Familiariza a los estudiantes con herramientas usadas en investigación (ej: Python para lógica simbólica).  

---



### **Ejemplo concreto**  
Un cuaderno podría incluir:  
1. Una explicación de *modus ponens* con una fórmula predefinida (`p → q`, `p`, luego `q`).  
2. Una celda interactiva donde el estudiante modifique `p` o `q` y observe cómo falla la regla si las premisas cambian.  
3. Un ejercicio para programar un verificador de *modus ponens* usando diccionarios de Python.  



Este enfoque transforma la lógica (a menudo abstracta) en una experiencia tangible y adaptable.

# Presentación de la estructura de la clase  

# Desarrollo de habilidades Metacognitivas en enseñanza con metodología IAE 



Desarrollar habilidades metacognitivas en los estudiantes dentro de una **Investigación Acción Educativa (IAE)** implica un proceso cíclico de reflexión, acción y evaluación. Aquí te propongo una estrategia estructurada en fases, alineada con la IAE, para fomentar la metacognición:

---



### **1. Diagnóstico Inicial (Fase de Observación)**  
- **Identifica el nivel metacognitivo actual**:  
  - Realiza cuestionarios, entrevistas o actividades reflexivas (ej.: "¿Cómo estudiaste para el último examen? ¿Qué te funcionó o no?").  
  - Observa si los estudiantes pueden explicar sus procesos de aprendizaje o identificar dificultades.  

- **Registra evidencias**: Anota cómo los estudiantes planifican, monitorean y evalúan sus tareas (ej.: diarios de aprendizaje, grabaciones de debates).  

---



### **2. Diseño de Intervenciones (Fase de Planificación)**

  
**a. Enseñanza explícita de estrategias metacognitivas**:  
  - **Modelado**: Muestra cómo *tú* piensas al resolver un problema ("Pensamiento en voz alta"). Ejemplo:  
    *"Primero, voy a leer el objetivo de la clase. Luego, revisaré si entiendo los conceptos clave..."*.  
  - **Listas de verificación (checklists)**: Proporciona guías para autoevaluarse (ej.: "¿Puedo explicar este tema con mis propias palabras?").  



**b. Herramientas para la autorregulación**:  
  - **Diarios de aprendizaje**: Pide que registren:  
    - *"¿Qué aprendí hoy?"* (conocimiento).  
    - *"¿Cómo lo aprendí?"* (proceso).  
    - *"¿Qué me falta por entender?"* (brechas).  
  - **Rúbricas de autoevaluación**: Incluye criterios como: *"Puedo resolver ejercicios sin ayuda"* o *"Sé dónde buscar información confiable"*.  

**c. Espacios de reflexión colaborativa**:  
  - **Debates metacognitivos**: En grupos, discuten: *"¿Qué estrategia usamos? ¿Funcionó? ¿Por qué?"*.  
  - **Peer feedback**: Intercambian comentarios sobre sus procesos (ej.: "Tú organizaste bien tus ideas, pero podrías revisar las fuentes").  

---



### **3. Implementación (Fase de Acción)**  
- **Integra la metacognición en las actividades cotidianas**:  
  - Antes de una tarea: *"¿Qué sabes ya sobre este tema? ¿Cómo planeas abordarlo?"*.  
  - Durante la tarea: *"¿Estás siguiendo tu plan? ¿Necesitas ajustarlo?"*.  
  - Después: *"¿Lograste el objetivo? ¿Qué cambiarías la próxima vez?"*.  
- **Usa preguntas clave**:  
  - *"¿Qué parte fue más difícil? ¿Por qué?"* (identificación de obstáculos).  
  - *"Si tuvieras que enseñarle esto a un compañero, ¿cómo lo harías?"* (transferencia).  

---

### **4. Evaluación y Reflexión (Fase de Observación/Reflexión)**  
- **Analiza el impacto**: Compara evidencias pre y post intervención (ej.: diarios, desempeño en tareas).  
- **Reflexión grupal**: Realiza una sesión donde los estudiantes compartan:  
  - *"¿Qué estrategias metacognitivas les ayudaron más?"*.  
  - *"¿Cómo se sintieron al gestionar su aprendizaje?"*.  
- **Ajusta la intervención**: Si notas que persisten dificultades, propón nuevas herramientas (ej.: mapas conceptuales para organizar ideas).  

---

### **5. Iteración (Ciclo de IAE)**  
Repite el ciclo con ajustes basados en los hallazgos. Por ejemplo:  
- Si los estudiantes no identifican errores, introduce actividades de *análisis de errores* ("¿Por qué te equivocaste? ¿Cómo corregirlo?").  
- Si les cuesta planificar, usa herramientas visuales como *diagramas de flujo* para secuenciar pasos.  

---

### **Ejemplo Práctico**  
**Situación**: Estudiantes no revisan sus errores en matemáticas.  
- **Intervención**:  
  1. **Modelado**: Resuelves un problema cometiendo un error adrede y muestras cómo detectarlo.  
  2. **Checklist**: "¿Revisé cada paso? ¿Mi respuesta tiene sentido?".  
  3. **Diario**: "Hoy cometí un error en... Lo corregí cambiando...".  

---

### **Claves para el Éxito**  
- **Consistencia**: Integra la metacognición en todas las clases, no como actividad aislada.  
- **Andamiaje**: Reduce gradualmente la guía del docente a medida que los estudiantes ganan autonomía.  
- **Cultura de error**: Normaliza los errores como parte del aprendizaje ("¿Qué podemos aprender de esto?").  

La metacognición no solo mejora el logro académico, sino que empodera a los estudiantes para ser aprendices autónomos y resilientes. En la IAE, este proceso se enriquece al ser colaborativo (docente-estudiantes) y basado en evidencia concreta.  

¿Te gustaría profundizar en alguna herramienta específica o ajustar la estrategia a un nivel educativo en particular?

# Calendario Académico Semestre 2025-2  






# Cursos que orienta el profesor Marco Julio Cañas Campillo en 2025  

1. Cálculo Vectorial para Ingeniería Agropecuaria
2. Análisis Numérico para Licenciatura en Matemáticas. 
3. Práctica Pedagógica V para Licenciatura en Educación Infantil
4. Fundamentos de Lógica para Licenciatura en Matemáticas 


# Horario de clases del profesor Marco

* Lunes 8-12: Cálculo Vectorial
* Martes 8-12 M: Análisis Numérico. 
* Miércoles 10 a 11 M: Machine Learnig
* Miércoles de 3 a 4 de la tarde: ARIMA
* Jueves 2 a 6 PM: Práctica Pedagógica V: Desarrollo del pensamiento matemático en   
  la infancia. 
* Viernes 8 - 12 M: Fundamentos de Lógica. 
* Sábados 8-12 Asesorías y espacio para retroalimentación y apoyo al trabajo independiente  
  y desarrollo de habilidades metacognitivas. 

# Coordinador de los cursos de la Facultad de Educación para regiones:    

Andrés Vélez: regioneducacion.fedu@udea.edu.co  
Coordinador Regiones  
Facultad de Educación  
Universidad de Antioquia  

## Monitores
* Manuel San Juan Serrano: Contactar escribiendo al correo: manuel.serrano1@udea.edu.co
* Yeifry Sebastián Uribe: Contactar escribiendo al correo: yeifry.uribe@udea.edu.co

## Referentes 

* [Jupyter Book de fundamentos_logica](file:///C:/Users/marco/Documentos/docencia/fundamentos_logica/fundamentos_logica_book/_build/html/index.html)


* [Decargue Crocodile Clip aquí](https://crocodileclips.net/descargar-crocodile-clips/)

* [Matemáticas discretas Una introducción abierta, 3ª edición](https://discrete.openmathbooks.org/dmoi3.html)  
  
* [Desarrollo del pensamiento matemático con calculadora Casio ](https://bibliotecadigital.udea.edu.co/entities/publication/17180405-9f1d-4800-aa7c-e6369779cece)

* [CALCULO I DE UNA VARIABLE Ron Larson-Bruce Edwards. Mc Graw Hill. 9º Edición](https://www.academia.edu/42139251/CALCULO_I_DE_UNA_VARIABLE_Ron_Larson_Bruce_Edwards_Mc_Graw_Hill_9o_Edici%C3%B3n)   
  

* [Grajales Vanegas, L. M., Restrepo Estrada, C. E., Restrepo Ochoa, S. I., & Ruíz De Villalba, F. (2015). Matemáticas I para las ciencias económicas.](https://bibliotecadigital.udea.edu.co/handle/10495/3010)
  
* R. Duval y Semiosis y pensamiento humano, 2.ª ed. Cali, Colombia: Programa Editorial Universidad del Valle, 2017. [En línea]. Disponible en: https://programaeditorial.univalle.edu.co/gpd-semiosis-y-pensamiento-humano-9789587655278-63324cdb0f6b3.html

* [Aylwin, C. U. (2011). Lógica, conjuntos y números. Universidad de los Andes, Consejo de Publicaciones, Colección: Ciencias Básicas, Serie: Matemáticas.](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://www.u-cursos.cl/ciencias/2011/1/MC110/1/material_docente/bajar?id_material=574722)
  
* [Chollet, F. (2021). Deep learning with Python. Simon and Schuster.](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://tanthiamhuat.wordpress.com/wp-content/uploads/2018/03/deeplearningwithpython.pdf)  
  
* [Watson, S., Stewart, J., & Redlin, L. (2009). Precálculo. Matemáticas para el cálculo.](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://students.aiu.edu/submissions/profiles/resources/onlineBook/k6L8A3_precalculo_-_matematicas_para_el_calculo-1.pdf)  

* [Purcell, E. J., Varberg, D., & Rigdon, S. E. (2007). Cálculo diferencial e integral. Pearson Educación.](https://github.com/marco-canas/calculo/blob/main/referents/purcell/purcell_calculo.pdf)

  

* [stewart cálculo](https://udeaeduco-my.sharepoint.com/:b:/g/personal/marco_canas_udea_edu_co/EZgXZjAp8QxPqOAim2hs6LcBNPLGjSHf-xwYnUVYkwa04w?e=RZdTCy)  


* [Recomendación de la UNESCO sobre ciencia abierta](https://unesdoc.unesco.org/ark:/48223/pf0000379949_spa)

* [chatGPT](https://openai.com/blog/chatgpt)  

* [Géron, A. (2017). Hands-on machine learning with scikit-learn and tensorflow: Concepts. Tools, and Techniques to build intelligent systems.](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/http://14.139.161.31/OddSem-0822-1122/Hands-On_Machine_Learning_with_Scikit-Learn-Keras-and-TensorFlow-2nd-Edition-Aurelien-Geron.pdf)   



* [McKinney, W. (2012). Python for data analysis: Data wrangling with Pandas, NumPy, and IPython. " O'Reilly Media, Inc.".](https://wesmckinney.com/book/) 

# Como estudiante, encuentro que...   

F: Mis Fortalezas son:     
O: Mis Oportunidades son:    
D: Mis Debilidades son:    
A: Lo que Amenazas mi aprendizaje es:  

### [Evaluamos al profesor Marco Cañas Aquí](https://forms.office.com/Pages/ResponsePage.aspx?id=IefhmYRxjkmK_7KtTlPBwkanXIs1i1FEujpsZgO6dXpUREJPV1kxUk1JV1ozTFJIQVNIQjY5WEY3US4u)

### Continue su aprendizaje en la siguiente clase a través del siguiente [vínculo]()

## Agradecimientos  

Doy gracias a Dios por la vida de mi Hijo Joseph Cañas Osorio y la madurez que ha alcanzado. Este hijo me enorgullece y me hace falta abrazarlo cada día. 

Y a mi esposa Yasmira Emperatriz Barboza Mogollón por su apoyo, orientación y acompañamiento. 